# Introduction to ChromaDB

## Why a Vector Database?

| | Relational DB (SQL) | Vector DB |
|---|---|---|
| **Data type** | Structured rows/columns | High-dimensional vectors |
| **Query type** | Exact match, range, join | Nearest-neighbour (similarity) |
| **Use case** | Invoices, users, orders | Embeddings, semantic search, face recognition |
| **Search** | `WHERE id = 42` | 'Find the 5 most similar embeddings to this query' |

A vector database stores embeddings and retrieves them **by similarity** — not by exact value. The CLIP embeddings we produced in the previous notebook are 512-D vectors. Once stored in a vector database, we can ask: *'Which enrolled face is most similar to this new photo?'* — and the database answers in milliseconds, even with millions of stored embeddings.

In this notebook we use **ChromaDB**, an open-source vector database written in Python that requires no external server and ships as a pip package.


## Core Concepts

| Term | Meaning |
|------|---------|
| **Client** | Entry point — manages the database connection and collections |
| **Collection** | A named group of embeddings (like a table in SQL) |
| **ID** | Unique string identifier for each row |
| **Embedding** | The vector stored in the collection |
| **Document** | Optional raw text associated with the embedding (useful for text tasks) |
| **Metadata** | Arbitrary key-value pairs stored alongside each embedding (e.g. timestamp, label) |

A single row in a collection = (ID, embedding, [document], [metadata]).

## Installation and Setup

In [ ]:
# pip install chromadb  (or: uv add chromadb)
import chromadb

print(f'ChromaDB version: {chromadb.__version__}')

ChromaDB version: 1.5.5


`EphemeralClient()` creates an in-memory database that exists only while the current Python process is alive. That is why you do not see database files on disk.

This is useful for teaching and quick experiments because it keeps the setup simple.

Use `PersistentClient(path='...')` when you want the collection to survive after the notebook stops running. In later application notebooks, persistence matters because enrolled faces or stored embeddings must still exist in a new session.

In [ ]:
# EphemeralClient: in-memory only — data is lost when the process ends.
# Use PersistentClient(path='./my_db') to save to disk.
#
# NOTE: EphemeralClient shares the same underlying in-memory store within a
# kernel session, so create_collection() raises InternalError if the cell is
# re-run. get_or_create_collection() is idempotent: creates on first run,
# returns the existing collection on subsequent runs.
client = chromadb.EphemeralClient()

# Create or retrieve a collection. distance_function controls how similarity is measured.
# 'cosine' is the right choice for normalised embeddings (e.g. CLIP).
collection = client.get_or_create_collection(
    name='demo_embeddings',
    metadata={'hnsw:space': 'cosine'}  # distance metric: cosine
)

print(f'Collection: {collection.name}')
print(f'Current count: {collection.count()}')


Collection: demo_embeddings
Current count: 0


## Adding Data

Use `collection.add()` to insert one or more rows.
Each row needs at minimum an **ID** and an **embedding**.

In [ ]:
import numpy as np

# We'll use synthetic 4-D embeddings for clarity.
# In a real system these would be 512-D CLIP outputs.

rng = np.random.default_rng(42)  # fixed seed for reproducibility

# Simulate face embeddings for 3 students
alice_emb  = rng.standard_normal(4).tolist()
bob_emb    = rng.standard_normal(4).tolist()
charlie_emb = rng.standard_normal(4).tolist()

collection.add(
    ids        = ['alice_001', 'bob_001', 'charlie_001'],
    embeddings = [alice_emb, bob_emb, charlie_emb],
    metadatas  = [
        {'name': 'Alice',   'session': 'enroll_2024',  'frame': 0},
        {'name': 'Bob',     'session': 'enroll_2024',  'frame': 0},
        {'name': 'Charlie', 'session': 'enroll_2024',  'frame': 0},
    ]
)

print(f'Collection now has {collection.count()} embeddings')

Collection now has 3 embeddings


In [ ]:
# Add more embeddings for Alice (e.g. from multiple captures at enrollment)
# Small noise simulates the same person photographed twice
alice_emb2 = (np.array(alice_emb) + rng.standard_normal(4) * 0.05).tolist()
alice_emb3 = (np.array(alice_emb) + rng.standard_normal(4) * 0.05).tolist()

collection.add(
    ids        = ['alice_002', 'alice_003'],
    embeddings = [alice_emb2, alice_emb3],
    metadatas  = [
        {'name': 'Alice', 'session': 'enroll_2024', 'frame': 1},
        {'name': 'Alice', 'session': 'enroll_2024', 'frame': 2},
    ]
)

print(f'Total embeddings: {collection.count()}')

Total embeddings: 5


## Querying by Similarity

`collection.query()` finds the **K nearest neighbours** of a query embedding.

ChromaDB returns **distances** (not similarities) — lower is more similar. For the cosine metric:

$$d_{\text{cosine}}(a, b) = 1 - \frac{a \cdot b}{\|a\| \cdot \|b\|}$$

- distance = 0 → vectors point in the same direction → perfect match
- distance = 1 → vectors are orthogonal → no similarity
- distance = 2 → vectors point in opposite directions (rare for real embeddings)

To convert back to similarity: $\text{similarity} = 1 - d$.


In [ ]:
# Simulate Alice presenting at verification: her embedding with small perturbation
alice_query = (np.array(alice_emb) + rng.standard_normal(4) * 0.1).tolist()

results = collection.query(
    query_embeddings = [alice_query],   # list of query vectors
    n_results = 3,                       # return top-3 most similar
    include = ['metadatas', 'distances'],
)

print('Top-3 results for Alice query:')
for rank, (doc_id, dist, meta) in enumerate(zip(
        results['ids'][0],
        results['distances'][0],
        results['metadatas'][0])):
    similarity = 1 - dist  # convert cosine distance to similarity
    print(f'  [{rank+1}] id={doc_id:12s}  cosine_similarity={similarity:.4f}  name={meta["name"]}')

Top-3 results for Alice query:
  [1] id=alice_003     cosine_similarity=0.9988  name=Alice
  [2] id=alice_002     cosine_similarity=0.9987  name=Alice
  [3] id=alice_001     cosine_similarity=0.9979  name=Alice


## Metadata Filtering

The `where` argument filters results by metadata **before** the similarity search.
This lets you search only within a specific group (e.g. a single student session).

In [ ]:
# Search only within Alice's enrollment entries
results_filtered = collection.query(
    query_embeddings = [alice_query],
    n_results = 2,
    where = {'name': 'Alice'},          # only entries where metadata 'name' == 'Alice'
    include = ['metadatas', 'distances'],
)

print('Filtered to Alice only:')
for doc_id, dist, meta in zip(
        results_filtered['ids'][0],
        results_filtered['distances'][0],
        results_filtered['metadatas'][0]):
    print(f'  id={doc_id:12s}  similarity={1-dist:.4f}  frame={meta["frame"]}')

Filtered to Alice only:
  id=alice_003     similarity=0.9988  frame=2
  id=alice_002     similarity=0.9987  frame=1


## Getting and Deleting Entries

In [ ]:
# Retrieve a specific entry by ID
result = collection.get(
    ids=['alice_001'],
    include=['embeddings', 'metadatas']
)
print('alice_001 metadata:', result['metadatas'][0])
print('alice_001 embedding first 4 values:', result['embeddings'][0][:4])

# Delete an entry
collection.delete(ids=['charlie_001'])
print(f'After deleting charlie_001: {collection.count()} entries remain')

alice_001 metadata: {'session': 'enroll_2024', 'frame': 0, 'name': 'Alice'}
alice_001 embedding first 4 values: [ 0.30471706 -1.03998411  0.75045115  0.94056463]
After deleting charlie_001: 4 entries remain


## How Similarity Search Works under the Hood

Brute-force similarity search scales as **O(N × D)**: for N=1M embeddings of dimension D=512 that is 512 million multiplications per query — too slow at scale.

ChromaDB uses **HNSW** (Hierarchical Navigable Small World), an Approximate Nearest Neighbour (ANN) graph algorithm.

**Intuition:**

1. Embeddings are organised into a multi-layer graph where each node connects to its nearest neighbours.
2. A query starts at the top (sparse) layer and greedily traverses the graph by always moving to the neighbour that is most similar to the query.
3. This narrows the search to a small candidate set, which is then re-ranked exactly.

The result is **approximate** (occasionally the true nearest neighbour might be missed) but **fast** — typical query time is O(log N) rather than O(N).

For teaching datasets with a few dozen embeddings, the difference is meaningless in practice. For production systems with millions of faces or documents, HNSW or similar algorithms (FAISS, ScaNN) are essential.

The `hnsw:space` parameter in the collection metadata must match the geometry your embeddings were trained with. CLIP vectors are L2-normalised (unit norm), which makes cosine distance and inner product equivalent — but the collection must still be told which one to use.


## Distance Metrics

When creating a collection, choose the metric that matches how your embeddings were trained:

| Metric | `hnsw:space` | Best for |
|--------|-------------|----------|
| Cosine distance | `'cosine'` | Normalised embeddings (CLIP, Sentence-BERT, most modern models) |
| L2 / Euclidean | `'l2'` | Raw feature vectors where magnitude matters |
| Inner product | `'ip'` | Maximum inner product search; faster when vectors are pre-normalised |

> For CLIP embeddings, use **cosine** — the model was trained with cosine similarity as the objective.

In [ ]:
# Demonstrate the three metric options (collections only, no data added)
client2 = chromadb.EphemeralClient()

col_cosine = client2.create_collection('test_cosine', metadata={'hnsw:space': 'cosine'})
col_l2     = client2.create_collection('test_l2',     metadata={'hnsw:space': 'l2'})
col_ip     = client2.create_collection('test_ip',     metadata={'hnsw:space': 'ip'})

print('Collections created:', [c.name for c in client2.list_collections()])

Collections created: ['test_ip', 'test_l2', 'demo_embeddings', 'test_cosine']


## Summary

| Operation | Code |
|-----------|------|
| Create client | `chromadb.EphemeralClient()` / `PersistentClient(path)` |
| Create collection | `client.create_collection(name, metadata={'hnsw:space': 'cosine'})` |
| Add embeddings | `collection.add(ids, embeddings, metadatas)` |
| Query by similarity | `collection.query(query_embeddings, n_results, where)` |
| Get by ID | `collection.get(ids)` |
| Delete by ID | `collection.delete(ids)` |
| Count entries | `collection.count()` |

The main idea is simple: embeddings become useful at scale only when we can retrieve their nearest neighbours efficiently.

That matters for face recognition, image retrieval, and also for NLP systems that store sentence or document embeddings. The next notebook combines detection, embeddings, and vector search into a full face-recognition pipeline.